# UABR-SLM · Notebook 10 — Self-RAG-adapted critique token `<REL>`/`<IRR>` (extension; validation only)

GPU **T4** (×1) · Internet not needed · ~1–1.5 h. Test stays sealed.

**Design (team decisions, 11 Nov 2026, fixed before any result)**

| Item | Value |
|---|---|
| Kept from Self-RAG | `[Retrieve]` = the existing `<ANSWER>`/`<CALL>` router; `[IsRel]` = new; `[IsSup]` merged into IsRel (for a one-entity answer "the window states the fact" is the same question); `[IsUse]` dropped (meaningless for one-word answers); per-passage answering + selection |
| Tokens / slot | `<REL>`, `<IRR>` appended (ids 12000, 12001; tokenizer file unchanged). Scored only, at the end of the single-window evidence prompt `<USER> q<ASSISTANT><CALL><EVIDENCE> w</EVIDENCE>`: z_REL − z_IRR. Answer format unchanged |
| Start | θ_route,preSFT (`06_routing/router_action.pt`) |
| Training data | **half-B** questions (never Q&A-trained, = router training set), non-overlapping top-10 windows each; label = subject AND gold alias in the window (automatic; G2: precision 79 %, recall 81 %); all positives + 3 negatives per positive (half hard: answer-only or subject-only) |
| Trainable | **transformer body + the 2 new rows**; old (tied) embedding rows frozen (R5); + half-A Q&A replay (answer tokens) + half-B router replay (2-logit decision) |
| Checkpoint rule | best critique AUROC on a 300-question validation **selection** subset (not in G2), among checkpoints whose closed EM is within 1 pp of θ_route and routing AUROC within 0.01 (same subset) |
| Gate G10 | on 500 other validation questions: closed AND evidence EM within 1 pp of θ_route; full-validation routing AUROC within 0.01 |
| RQ-S1 | IsRel AUROC on the 600 **G2** windows (AI-drafted labels, team blind check pending — report only after it) and on validation windows with automatic labels. **Comparator = best answer-free baseline** (BM25 score, BM25 rank, subject mention, own article, lexical logistic regression, θ_route hidden-state probe; best chosen on the selection subset). The gold-using rule is a label-quality reference only. **Rule (fixed):** PASS iff critic − best baseline on G2 has a clustered 95 % CI lower bound > 0; else report "a 40M model does not critique beyond lexical matching" |
| RQ-S2 | evidence EM / retrieval damage: (a) top-3 concatenated, (b) per-window answers from the top-5, keep max P(`<REL>`), (b2) as (b) but answer closed-book when max P(`<REL>`) < 0.5, (c) same selection by the best baseline, (c1) BM25 top-1 window, (d) best of 5 (upper bound). Computed always; interpreted only if RQ-S1 passes |
| RQ-S3 | notebook 07 SFT recipe from the critic model (seeds 11, 29): drop in IsRel AUROC vs drop in routing AUROC, flips by margin |

**Inputs:** outputs of notebooks **00, 01, 02, 05, 06 (v2), 07**; the G2 file comes from the repo (`data/audit/g2_support_audit_labeled.csv`).

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from uabr.data import tokenizer as T
from uabr.data import qa_format as Q
from uabr.eval import prompts as PR
from uabr.eval.batched import greedy_batch
from uabr.eval.generate import load_model
from uabr.eval.metrics import alias_em, alias_contains
from uabr.retrieval.select import select_non_overlapping
from uabr.routing.metrics import auroc, clustered_bootstrap_auroc
from uabr.routing.pilot import paired_cluster_bootstrap, flips_by_margin
from uabr.routing.router import router_prompt, router_scores
from uabr.critic.selfrag import add_critique_tokens, critique_ids, critique_scores, last_hidden, train_critic
from uabr.train.sft import STANDARD, instruction_batches, train_sft

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/10_selfrag" if ON_KAGGLE else os.path.abspath("outputs/10_selfrag")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4"
SEED = 1010
K_TRAIN, K_EVAL, NEG_PER_POS = 10, 5, 3
C_LR, C_EPOCHS, C_BS, C_EVAL = 3e-5, int(os.environ.get("C_EPOCHS", 3)), 32, int(os.environ.get("C_EVAL", 100))
TOL_PP, TOL_AUROC = 1.0, 0.01
SEL_N, PRES_N = int(os.environ.get("SEL_N", 300)), int(os.environ.get("PRES_N", 500))
SEEDS = [int(s) for s in os.environ.get("SEEDS", "11,29").split(",")]
N_BOOT = int(os.environ.get("N_BOOT", 1000))
MAX_NEW, SEQ = 16, 512
LIMIT = int(os.environ.get("LIMIT", 0))

def one_input(pattern, env=None):
    if env and os.environ.get(env): return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert len(hits) == 1, f"{pattern}: expected exactly 1 attached file, found {len(hits)}: {hits}"
    return hits[0]

popqa = pd.read_json(find_input("00_popqa_splits/popqa_prepared.jsonl", "POPQA_PREPARED"), lines=True).set_index("id", drop=False)
tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20")).set_index("id")
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
halfA_ids = set(int(x) for x in open(find_input("05_mixed/manifests/halfA_ids.txt", "HALFA")).read().split())
POLICY_PATH = one_input("06_routing/routing_policy.json", "POLICY"); D06 = os.path.dirname(POLICY_PATH)
assert json.load(open(POLICY_PATH))["format"] == "action"
ROUTER = os.path.join(D06, "router_action.pt")
V = pd.read_parquet(os.path.join(D06, "labels_val.parquet")); B = pd.read_parquet(os.path.join(D06, "labels_halfB.parquet"))
REPORT07 = one_input("07_sft_pilot/pilot_report.json", "REPORT07"); D07 = os.path.dirname(REPORT07)
G2_PATH = os.environ.get("G2") or os.path.join(CODE_ROOT, "data", "audit", "g2_support_audit_labeled.csv")
G2 = pd.read_csv(G2_PATH)
if LIMIT: V, B = V.head(LIMIT), B.head(LIMIT)
V, B = V.reset_index(drop=True), B.reset_index(drop=True)
WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]

g2_ids = set(G2.id.astype(int))
rng = np.random.default_rng(SEED)
pool = V[~V.id.isin(g2_ids)]
SELQ = pool.sample(min(SEL_N, len(pool) // 3), random_state=SEED)                  # checkpoint selection (never evaluated on)
EVALQ = V[~V.id.isin(SELQ.id)].reset_index(drop=True)
PRESQ = EVALQ[~EVALQ.id.isin(g2_ids)].sample(min(PRES_N, len(EVALQ)), random_state=SEED + 1)
print(f"router {ROUTER} | half-B {len(B):,} | val {len(V):,} -> selection {len(SELQ)}, evaluation {len(EVALQ):,} (preservation subset {len(PRESQ)}) | G2 {len(G2)} questions")

## 3 · Window items with automatic labels, the G2 windows, the critic training set

In [ ]:
def items_for(qdf, k, raw=False):
    """One row per (question, window). raw=True: the first k raw BM25 positions (as in the G2 audit);
    else the non-overlapping top-k (as used for answering)."""
    rows = []
    for r in qdf.itertuples(index=False):
        t = top.loc[r.id]; pq = popqa.loc[r.id]; docs = list(t.doc_ids); pos = {int(d): j for j, d in enumerate(docs)}
        chosen = [int(d) for d in docs[:k] if d >= 0] if raw else select_non_overlapping(docs, WART, WSTART, k=k, window_words=60)[0]
        for rank, d in enumerate(chosen):
            j = pos[int(d)]
            rows.append({"id": int(r.id), "group_id": pq.group_id, "prop": pq.prop, "doc_id": int(d),
                         "rank": rank, "bm25_rank": j, "bm25_score": float(t.scores[j]), "answer_in": bool(t.answer_in[j]),
                         "subject_in": bool(t.subject_in[j]), "own_article": bool(t.own_article[j]),
                         "auto_rel": int(bool(t.answer_in[j]) and bool(t.subject_in[j])), "question": pq.question})
    df = pd.DataFrame(rows)
    df["prompt"] = [Q.evidence_prompt_ids(tok, q, [WTEXT[d]], SEQ - MAX_NEW)[0] for q, d in zip(df.question, df.doc_id)]
    assert all(p[-2] != T.EVIDENCE_ID for p in df.prompt), "a window did not fit"
    return df

TRAIN_ALL = items_for(B, K_TRAIN)
SEL_IT = items_for(SELQ, K_EVAL)
EVAL_IT = items_for(EVALQ, K_EVAL)

# G2: 200 pilot questions x raw top-3 (exactly the audited windows); labels AI-drafted (team check pending)
g2 = G2[G2.subject.fillna("").str.strip() != ""]
G2_IT = items_for(popqa.loc[g2.id.astype(int)], 3, raw=True)
lab = {(int(r.id), j): (int(getattr(r, f"manual_supports_{j+1}")), int(getattr(r, f"uncertain_{j+1}")), str(getattr(r, f"window_{j+1}")))
       for r in g2.itertuples(index=False) for j in range(3)}
G2_IT["g2_label"] = [lab[(i, j)][0] for i, j in zip(G2_IT.id, G2_IT["rank"])]
G2_IT["uncertain"] = [lab[(i, j)][1] for i, j in zip(G2_IT.id, G2_IT["rank"])]
match = np.mean([WTEXT[d].strip() == lab[(i, j)][2].strip() for i, j, d in zip(G2_IT.id, G2_IT["rank"], G2_IT.doc_id)])
print(f"G2 windows {len(G2_IT)} (questions {G2_IT.id.nunique()}) | text identical to the audited window: {match:.1%} | "
      f"relevant {G2_IT.g2_label.mean():.1%} | auto rule vs G2 labels: precision {G2_IT[G2_IT.auto_rel == 1].g2_label.mean():.1%}, "
      f"recall {G2_IT[G2_IT.g2_label == 1].auto_rel.mean():.1%}")
assert match > 0.98, "G2 windows do not match the retrieval cache - wrong 02 output attached?"

# training set: all positives + NEG_PER_POS negatives per positive, half hard (answer-only / subject-only)
pos = TRAIN_ALL[TRAIN_ALL.auto_rel == 1]; neg = TRAIN_ALL[TRAIN_ALL.auto_rel == 0]
hard = neg[neg.answer_in ^ neg.subject_in]; easy = neg.drop(hard.index)
n_neg = NEG_PER_POS * len(pos); n_hard = min(len(hard), n_neg // 2)
TRAIN = pd.concat([pos, hard.sample(n_hard, random_state=SEED), easy.sample(min(len(easy), n_neg - n_hard), random_state=SEED)]).sample(frac=1, random_state=SEED)
print(f"half-B windows {len(TRAIN_ALL):,} (relevant {TRAIN_ALL.auto_rel.mean():.1%}) -> training {len(TRAIN):,}: "
      f"{len(pos):,} relevant, {n_hard:,} hard + {len(TRAIN) - len(pos) - n_hard:,} other irrelevant | "
      f"selection windows {len(SEL_IT):,} | evaluation windows {len(EVAL_IT):,} (relevant {EVAL_IT.auto_rel.mean():.1%})")

## 4 · Answer-free baselines (fitted on the half-B training windows; best chosen on the selection subset)

In [ ]:
LEX = ["bm25_score", "bm25_rank", "subject_in", "own_article"]
lexX = lambda d: d[LEX].astype(float).to_numpy()
lex_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(lexX(TRAIN), TRAIN.auto_rel)
m0 = load_model(ROUTER, DEV)
H = {k: last_hidden(m0, d.prompt.tolist(), DEV) for k, d in [("train", TRAIN), ("sel", SEL_IT), ("eval", EVAL_IT), ("g2", G2_IT)]}
probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1)).fit(H["train"], TRAIN.auto_rel)
del m0
def baselines(d, h):
    return {"bm25_score": d.bm25_score.to_numpy(float), "bm25_rank": -d.bm25_rank.to_numpy(float),
            "subject_in": d.subject_in.to_numpy(float) + 1e-3 * d.bm25_score.to_numpy(float) / (1 + d.bm25_score.max()),
            "own_article": d.own_article.to_numpy(float) + 1e-3 * d.bm25_score.to_numpy(float) / (1 + d.bm25_score.max()),
            "lexical_lr": lex_lr.predict_proba(lexX(d))[:, 1], "hidden_probe": probe.predict_proba(h)[:, 1]}
for k, d in [("sel", SEL_IT), ("eval", EVAL_IT), ("g2", G2_IT)]:
    for name, s in baselines(d, H[k]).items(): d[f"b_{name}"] = s
BASE = ["bm25_score", "bm25_rank", "subject_in", "own_article", "lexical_lr", "hidden_probe"]
sel_auc = {b: auroc(SEL_IT.auto_rel, SEL_IT[f"b_{b}"]) for b in BASE}
BEST = max(sel_auc, key=sel_auc.get)
print("selection-subset AUROC (auto labels):", {k: round(v, 3) for k, v in sel_auc.items()}, "| best answer-free baseline:", BEST)
print("(subject_in / own_article are binary rules; BM25 score only breaks ties)")

## 5 · Train the critique token from θ_route,preSFT

In [ ]:
def bans(model):
    return (T.CALL_ID,) + (critique_ids(model) if model.cfg.vocab_size > BASE_VOCAB else ())
def closed_em(model, qdf):
    g = greedy_batch(model, [Q.closed_prompt_ids(tok, q) for q in qdf.question], DEV, MAX_NEW, banned=bans(model), stop_ids=stop_ids)
    return float(np.mean([alias_em(PR.clean_generation(tok.decode(x)), a) for x, a in zip(g, qdf.gold_aliases)]))
def evidence_prompts(qdf, k=3):
    out = []
    for r in qdf.itertuples(index=False):
        sel, _ = select_non_overlapping(list(top.loc[r.id].doc_ids), WART, WSTART, k=k, window_words=60)
        out.append(Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)[0])
    return out
def evidence_em(model, qdf):
    g = greedy_batch(model, evidence_prompts(qdf), DEV, MAX_NEW, banned=bans(model), stop_ids=stop_ids)
    return float(np.mean([alias_em(PR.clean_generation(tok.decode(x)), a) for x, a in zip(g, qdf.gold_aliases)]))
def route_auroc(model, qdf):
    s, _ = router_scores(model, [router_prompt(tok, q, "action") for q in qdf.question], "action", DEV)
    return auroc(qdf.y, s)

m0 = load_model(ROUTER, DEV); BASE_VOCAB = m0.cfg.vocab_size
REF = {"sel_closed": closed_em(m0, SELQ), "sel_route": route_auroc(m0, SELQ), "pres_closed": closed_em(m0, PRESQ),
       "pres_evidence": evidence_em(m0, PRESQ), "val_route": route_auroc(m0, V)}
print("theta_route reference:", {k: round(v, 4) for k, v in REF.items()})
critic = add_critique_tokens(m0, seed=SEED); del m0
N_OLD = critic.cfg.vocab_size - 2

Arows = popqa[popqa.id.isin(halfA_ids)]
if LIMIT: Arows = Arows.head(LIMIT)
QA = []
for r in Arows.itertuples(index=False):
    QA.append(Q.closed_example(tok, r.question, r.answer))
    sel, _ = select_non_overlapping(list(top.loc[r.id].doc_ids), WART, WSTART, k=3, window_words=60)
    QA.append(Q.evidence_example(tok, r.question, [WTEXT[d] for d in sel], r.answer, SEQ)[0])
RR = ([router_prompt(tok, q, "action") for q in B.question], [int(y) for y in B.y])

def evaluate(model):
    a = auroc(SEL_IT.auto_rel, critique_scores(model, SEL_IT.prompt.tolist(), DEV))
    e, r = closed_em(model, SELQ), route_auroc(model, SELQ)
    ok = (e >= REF["sel_closed"] - TOL_PP / 100) and (r >= REF["sel_route"] - TOL_AUROC)
    return a, ok, {"closed_em_sel": round(e, 4), "route_auroc_sel": round(r, 4)}

stage(f"training critic on {len(TRAIN):,} windows | Q&A replay {len(QA):,} | router replay {len(RR[0]):,}")
best, hist = train_critic(critic, TRAIN.prompt.tolist(), TRAIN.auto_rel.tolist(), DEV, N_OLD, lr=C_LR, epochs=C_EPOCHS, bs=C_BS,
                          eval_every=C_EVAL, seed=SEED, log=stage, qa_replay=QA, router_replay=RR, evaluate=evaluate)
critic.load_state_dict(best["state"])
CRITIC = f"{OUT}/critic.pt"
torch.save({"model": best["state"], "model_cfg": critic.cfg.to_dict(), "rel_id": critique_ids(critic)[0], "irr_id": critique_ids(critic)[1],
            "best_step": best["step"], "info": best["info"]}, CRITIC)
pd.DataFrame(hist).to_csv(f"{OUT}/critic_train_history.csv", index=False)
display(pd.DataFrame(hist).round(4))
G10 = {"pres_closed": closed_em(critic, PRESQ), "pres_evidence": evidence_em(critic, PRESQ), "val_route": route_auroc(critic, V)}
G10_OK = (G10["pres_closed"] >= REF["pres_closed"] - TOL_PP / 100 and G10["pres_evidence"] >= REF["pres_evidence"] - TOL_PP / 100
          and G10["val_route"] >= REF["val_route"] - TOL_AUROC)
print(f"best step {best['step']} | selection AUROC {best['score']:.4f} | G10 preservation: closed {REF['pres_closed']:.1%} -> {G10['pres_closed']:.1%}, "
      f"evidence {REF['pres_evidence']:.1%} -> {G10['pres_evidence']:.1%}, routing AUROC {REF['val_route']:.4f} -> {G10['val_route']:.4f} | {'PASS' if G10_OK else 'FAIL'}")

## 6 · RQ-S1: can the critic judge relevance beyond answer-free cues?

In [ ]:
for d in (EVAL_IT, G2_IT): d["critic"] = critique_scores(critic, d.prompt.tolist(), DEV)
def auc_table(d, label):
    rows = []
    for name in ["critic"] + [f"b_{b}" for b in BASE]:
        ci = clustered_bootstrap_auroc(d[label].values, d[name].values, d.group_id.values, n_boot=N_BOOT)
        rows.append({"scorer": name, "auroc": ci["auroc"], "lo": ci["lo"], "hi": ci["hi"]})
    return pd.DataFrame(rows)
T_G2 = auc_table(G2_IT, "g2_label"); T_EV = auc_table(EVAL_IT, "auto_rel")
print(f"G2 windows (AI-drafted labels, n={len(G2_IT)}, relevant {G2_IT.g2_label.sum()}):"); display(T_G2.round(3))
print(f"validation windows (automatic labels, n={len(EVAL_IT):,}):"); display(T_EV.round(3))
def diff(d, label, a, b):
    f = lambda idx, w: auroc(d[label].values[idx], d[a if w == "a" else b].values[idx])
    return paired_cluster_bootstrap(f, d.group_id.values, n_boot=N_BOOT, seed=SEED)
S1_G2 = diff(G2_IT, "g2_label", "critic", f"b_{BEST}"); S1_EV = diff(EVAL_IT, "auto_rel", "critic", f"b_{BEST}")
RQ_S1 = bool(S1_G2["ci_lo"] > 0)
print(f"critic - best baseline ({BEST}): G2 {S1_G2['diff']:+.3f} [{S1_G2['ci_lo']:+.3f}, {S1_G2['ci_hi']:+.3f}] | "
      f"validation auto {S1_EV['diff']:+.3f} [{S1_EV['ci_lo']:+.3f}, {S1_EV['ci_hi']:+.3f}]")
ref = G2_IT.auto_rel
print(f"label-quality reference (uses the gold answer): subject AND alias rule on G2 -> AUROC {auroc(G2_IT.g2_label, ref):.3f}")
sc = G2_IT[G2_IT.answer_in]
print(f"shortcut test (G2 windows containing an answer alias, n={len(sc)}, of which {int((sc.g2_label == 0).sum())} do NOT state the fact): "
      f"critic AUROC {auroc(sc.g2_label, sc.critic) if sc.g2_label.nunique() == 2 else float('nan'):.3f} | "
      f"{BEST} {auroc(sc.g2_label, sc[f'b_{BEST}']) if sc.g2_label.nunique() == 2 else float('nan'):.3f}")
print("sensitivity, uncertain G2 windows dropped:", round(auroc(G2_IT[G2_IT.uncertain == 0].g2_label, G2_IT[G2_IT.uncertain == 0].critic), 3))
per_rel = EVAL_IT.groupby("prop").apply(lambda g: pd.Series({"n": len(g), "relevant": g.auto_rel.mean(),
            "critic": auroc(g.auto_rel, g.critic) if g.auto_rel.nunique() == 2 else np.nan,
            BEST: auroc(g.auto_rel, g[f"b_{BEST}"]) if g.auto_rel.nunique() == 2 else np.nan}))
display(per_rel.round(3))
print(f"RQ-S1: {'PASS - the critic beats the best answer-free baseline on G2' if RQ_S1 else 'NOT SUPPORTED - report: a 40M model does not critique beyond lexical/answer-free cues (RQ-S2 is not interpreted)'}")

## 7 · RQ-S2: does per-window selection help answering? (validation evaluation questions)

In [ ]:
g = greedy_batch(critic, EVAL_IT.prompt.tolist(), DEV, MAX_NEW, banned=bans(critic), stop_ids=stop_ids)
EVAL_IT["pred"] = [PR.clean_generation(tok.decode(x)) for x in g]
gold = dict(zip(EVALQ.id, EVALQ.gold_aliases))
EVAL_IT["em"] = [alias_em(p, gold[i]) for p, i in zip(EVAL_IT.pred, EVAL_IT.id)]
gc = greedy_batch(critic, [Q.closed_prompt_ids(tok, q) for q in EVALQ.question], DEV, MAX_NEW, banned=bans(critic), stop_ids=stop_ids)
ge = greedy_batch(critic, evidence_prompts(EVALQ), DEV, MAX_NEW, banned=bans(critic), stop_ids=stop_ids)
R2 = pd.DataFrame({"id": EVALQ.id, "group_id": EVALQ.group_id,
                   "closed": [alias_em(PR.clean_generation(tok.decode(x)), a) for x, a in zip(gc, EVALQ.gold_aliases)],
                   "a_top3_concat": [alias_em(PR.clean_generation(tok.decode(x)), a) for x, a in zip(ge, EVALQ.gold_aliases)]})
pick = lambda col: EVAL_IT.loc[EVAL_IT.groupby("id")[col].idxmax()].set_index("id")
pc, pb = pick("critic"), pick(f"b_{BEST}")
R2["b_critic_select"] = R2.id.map(pc.em)
P_REL = 1 / (1 + np.exp(-R2.id.map(pc.critic)))
R2["b2_critic_gated"] = np.where(P_REL >= 0.5, R2.b_critic_select, R2.closed)
R2["c_baseline_select"] = R2.id.map(pb.em)
R2["c1_bm25_top1"] = R2.id.map(EVAL_IT[EVAL_IT["rank"] == 0].set_index("id").em)
R2["d_best_of_5"] = R2.id.map(EVAL_IT.groupby("id").em.max())
R2 = R2.fillna(0)
CONDS = ["a_top3_concat", "b_critic_select", "b2_critic_gated", "c_baseline_select", "c1_bm25_top1", "d_best_of_5"]
summ = pd.DataFrame({c: {"EM": R2[c].mean(), "retrieval_damage": float(((R2.closed == 1) & (R2[c] == 0)).mean())} for c in CONDS}).T
print(f"closed-book EM {R2.closed.mean():.1%} | gated: answered closed-book for {(P_REL < 0.5).mean():.1%} of questions"); display(summ.round(4))
S2 = {}
for a, b in [("b_critic_select", "a_top3_concat"), ("b_critic_select", "c_baseline_select"), ("b2_critic_gated", "a_top3_concat")]:
    S2[f"{a} - {b}"] = paired_cluster_bootstrap(lambda idx, w: R2[a if w == "a" else b].values[idx].mean(), R2.group_id.values, n_boot=N_BOOT, seed=SEED)
    r = S2[f"{a} - {b}"]; print(f"{a} - {b}: {r['diff']*100:+.2f} pp [{r['ci_lo']*100:+.2f}, {r['ci_hi']*100:+.2f}]")
if not RQ_S1: print("(RQ-S1 did not pass: these numbers are reported but not interpreted, rule fixed in advance)")

## 8 · RQ-S3: does the critique token survive plain SFT? (notebook 07 recipe from the critic model)

In [ ]:
sx = pd.read_parquet(os.path.join(D07, "sft_examples.parquet"))
mk = lambda r: (list(map(int, r.tokens)), [False] * int(r.n_prompt) + [True] * (len(r.tokens) - int(r.n_prompt)))
SFT_TRAIN = [mk(r) for r in sx[sx.part == "train"].itertuples(index=False)]
POST = {}
for seed in SEEDS:
    out = f"{OUT}/sft_s{seed}"
    prev = sorted(glob.glob(f"/kaggle/input/**/10_selfrag/sft_s{seed}/final.pt", recursive=True))
    if prev: POST[seed] = prev[0]; stage(f"seed {seed}: reusing {prev[0]}"); continue
    if not os.path.exists(f"{out}/final.pt"):
        m = load_model(CRITIC, DEV)
        s = train_sft(m, instruction_batches(SFT_TRAIN, STANDARD["batch_size"], seed), dict(STANDARD, log_every_steps=250), seed, out, log=stage)
        stage(f"seed {seed}: {s['updates']} updates in {s['minutes']} min"); del m
    POST[seed] = f"{out}/final.pt"
rp = [router_prompt(tok, q, "action") for q in V.question]
def scores(path):
    m = load_model(path, DEV)
    r = {"g2": critique_scores(m, G2_IT.prompt.tolist(), DEV), "eval": critique_scores(m, EVAL_IT.prompt.tolist(), DEV),
         "route": router_scores(m, rp, "action", DEV)[0]}
    del m; return r
S0 = {"g2": G2_IT.critic.values, "eval": EVAL_IT.critic.values, "route": router_scores(critic, rp, "action", DEV)[0]}
RET = []
for seed, path in POST.items():
    S1 = scores(path)
    for name, d, lab, key in [("IsRel (G2 labels)", G2_IT, "g2_label", "g2"), ("IsRel (val auto labels)", EVAL_IT, "auto_rel", "eval"),
                              ("Retrieve / routing (val)", V, "y", "route")]:
        yv, gv = d[lab].values, d.group_id.values
        f = lambda idx, w: auroc(yv[idx], (S1 if w == "a" else S0)[key][idx])
        r = paired_cluster_bootstrap(f, gv, n_boot=N_BOOT, seed=seed)
        a0, a1 = auroc(yv, S0[key]), auroc(yv, S1[key])
        fl = flips_by_margin(S0[key], S1[key], np.abs(S0[key]), gv)
        RET.append({"seed": seed, "token": name, "auroc_0": a0, "auroc_100": a1, "diff": r["diff"], "lo": r["ci_lo"], "hi": r["ci_hi"],
                    "rel_drop_of_skill": (a0 - a1) / (a0 - 0.5) if a0 > 0.55 else np.nan, "flip_rate": fl["flip_rate"],
                    "auroc_small_margin_predicts_flip": fl["auroc_small_margin_predicts_flip"]["auroc"],
                    "default_positive_rate_0": float((S0[key] > 0).mean()), "default_positive_rate_100": float((S1[key] > 0).mean())})
RET = pd.DataFrame(RET); RET.to_csv(f"{OUT}/rq_s3_retention.csv", index=False); display(RET.round(3))
print("rel_drop_of_skill = (AUROC_0 - AUROC_100) / (AUROC_0 - 0.5): share of above-chance skill lost; compare IsRel vs Retrieve")

## 9 · Save + checks

In [ ]:
for name, d in [("eval_windows", EVAL_IT), ("g2_windows", G2_IT)]:
    d.drop(columns=["prompt"]).to_parquet(f"{OUT}/{name}.parquet", index=False)
R2.to_parquet(f"{OUT}/rq_s2_answers.parquet", index=False)
T_G2.to_csv(f"{OUT}/rq_s1_g2_auroc.csv", index=False); T_EV.to_csv(f"{OUT}/rq_s1_val_auroc.csv", index=False); summ.to_csv(f"{OUT}/rq_s2_summary.csv")
report = {"notebook": "10_selfrag", "rq_s1_pass": RQ_S1, "best_baseline": BEST, "selection_auroc": sel_auc, "rq_s1_g2": S1_G2, "rq_s1_val": S1_EV,
          "rq_s2": S2, "g10": {"ref": REF, "critic": G10, "pass": G10_OK}, "critic_best_step": best["step"], "critic_info": best["info"],
          "train": {"windows": len(TRAIN), "relevant": int(TRAIN.auto_rel.sum())}, "g2_text_match": match,
          "g2_labels": "AI-drafted (Claude), team blind check pending", "seeds": SEEDS, "router": ROUTER,
          "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}}
json.dump(report, open(f"{OUT}/selfrag_report.json", "w"), indent=1, default=float)
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("G10_answers_and_routing_preserved", G10_OK, f"closed {G10['pres_closed']:.1%} (ref {REF['pres_closed']:.1%}), evidence {G10['pres_evidence']:.1%} "
    f"(ref {REF['pres_evidence']:.1%}), routing AUROC {G10['val_route']:.4f} (ref {REF['val_route']:.4f})")
add("critic_checkpoint_qualified", not best["info"].get("unqualified", False), f"best step {best['step']}")
add("g2_windows_match_cache", match > 0.98, f"{match:.1%}")
add("critic_learned_something", auroc(EVAL_IT.auto_rel, EVAL_IT.critic) > 0.6, f"val auto AUROC {auroc(EVAL_IT.auto_rel, EVAL_IT.critic):.3f}", "warn")
add("g2_labels_verified_by_team", False, "AI-drafted labels: run the 50-row blind check before reporting RQ-S1", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "selfrag_extension", "status": status, "rq_s1_pass": RQ_S1, "checks": checks}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print(f"\nNOTEBOOK 10: {status} | RQ-S1 {'PASS' if RQ_S1 else 'not supported'}")